In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torch.nn.functional as F
from torchsummary import summary
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.feature_extraction import create_feature_extractor
import torch.nn as nn

from PIL import Image

import lpips

from diffusers.models import AutoencoderKL

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score, RocCurveDisplay
import seaborn as sns

from tqdm import tqdm
import copy
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
class SafeSubset(Dataset):
    """
    Envuelve cualquier Subset (o Dataset) y garantiza que cada tensor
    devuelto es contiguo en memoria propia, evitando el error
    'Trying to resize storage that is not resizable'.
    """
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return img.clone().contiguous(), label


In [ ]:
test_dir = '/workspace/dataset/test/'
train_dir = '/workspace/dataset/train/'

IMG_SIZE   = 256
BATCH_SIZE = 64
VAL_SPLIT  = 0.2
SEED       = 42
NITS     = 50
LR        = 1e-3

# Solo resize y normalización — para val y test (sin augmentation)
resize_only = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Resize + augmentation — solo para train
resize_augment = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=36),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomAutocontrast(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
train_ds_augmented = datasets.ImageFolder(root=train_dir, transform=resize_augment)
train_ds_clean     = datasets.ImageFolder(root=train_dir, transform=resize_only)
test_ds            = datasets.ImageFolder(root=test_dir,  transform=resize_only)

print(f"Clases detectadas : {train_ds_augmented.classes}")
print(f"class_to_idx      : {train_ds_augmented.class_to_idx}")

n_total = len(train_ds_augmented)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(
    range(n_total), [n_train, n_val], generator=generator
)

real_label = train_ds_augmented.class_to_idx['real']

real_train_indices = [
    i for i in train_indices.indices
    if train_ds_augmented.samples[i][1] == real_label
]
real_val_indices = [
    i for i in val_indices.indices
    if train_ds_clean.samples[i][1] == real_label
]

train_dataset = Subset(train_ds_augmented, train_indices.indices)
val_dataset   = Subset(train_ds_clean,     val_indices.indices)

train_real_dataset = Subset(train_ds_augmented, real_train_indices)
val_real_dataset   = Subset(train_ds_clean,     real_val_indices)

print(f"Tamaños — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_ds)}")
print(f"Tamaños reales — train_real: {len(train_real_dataset)}, val_real: {len(val_real_dataset)}")


In [ ]:
img, lbl = train_real_dataset[15]
print(f"Shape imagen    : {img.shape}")
print(f"Rango píxeles   : {img.min():.2f} a {img.max():.2f}")
print(f"Categoría       : {lbl} ({train_ds_augmented.classes[lbl]})")
print(f"Clases → índice : {test_ds.class_to_idx}")


In [ ]:
from torch.utils.data import Dataset

class CachedDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.cache = {}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        if idx not in self.cache:
            self.cache[idx] = self.base[idx]
        return self.cache[idx]

# Uso
train_real_dataset = CachedDataset(train_real_dataset)
val_real_dataset = CachedDataset = CachedDataset(val_real_dataset)


In [ ]:
dataloader_test = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

dataloader_train_real = DataLoader(
    train_real_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

dataloader_val_real = DataLoader(
    val_real_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Batches — test: {len(dataloader_test)}, test: {len(dataloader_test)}")
print(f"Batches — train_real: {len(dataloader_train_real)}, train_real: {len(dataloader_train_real)}")
print(f"Batches — val_real: {len(dataloader_val_real)}, val_real: {len(dataloader_val_real)}")

In [ ]:
import torch
torch.cuda.empty_cache()

print(f"VRAM ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reservada: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
device = torch.device("cuda")
resnet = resnet50(weights=ResNet50_Weights.DEFAULT).to(device)

In [ ]:
# Cargar el modelo pre-entrenado y ponerlo en modo inferencia
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
resnet.eval()

# Definir diccionario con los nombres de las capas que nos interesan
return_nodes = {
    'layer2': 'features_capa2',
    'layer3': 'features_capa3'
}

# Crear el extractor de características
feature_extractor = create_feature_extractor(resnet, return_nodes=return_nodes)

In [ ]:
class Extractor(nn.Module):

    def __init__(self):
        super().__init__()

        # Crear instancia del "feature extractor"
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        resnet.eval()
        return_nodes = {
            'layer2': 'features_capa2',
            'layer3': 'features_capa3'
        }
        self.feature_extractor = create_feature_extractor(resnet, return_nodes=return_nodes)

        # Capa de pooling (para homogeneizar los features antes de combinarlos)
        self.smoothing_pool = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)

        # Campos de salida
        self.campos_salida = list(return_nodes.values()) # ['layer2', 'layer3']

    def forward(self, x):
        """
        1. Extrae los features de las capas 2 y 3 (512x28x28, 1024x14x14)
        2. Ajusta el tamaño del segundo "feature" a 1024x28x28
        3. Concatena los dos features en un solo tensor
        """

        # 1. Extraer los features
        features = self.feature_extractor(x)

        # Convertir el diccionario de "features" a una lista
        features = [features[key] for key in self.campos_salida]

        # Suavizar los features para homogeneizarlos ya que vienen de
        # capas diferentes
        features_suav = [self.smoothing_pool(feat) for feat in features]

        # Extraer alto y ancho del primer feature (28x28)
        feat_h, feat_w = features_suav[0].shape[-2:]

        # 2. Ajustar el tamaño del segundo feature a 1024xfeat_hxfeat_w (1024x28x28)
        resizer = nn.AdaptiveAvgPool2d((feat_h, feat_w))

        # Combinar features en un listado
        features_res = [resizer(feat) for feat in features_suav]

        # 3. Concatenar los features
        features_finales = torch.cat(features_res, dim=1)

        return features_finales

In [ ]:
# Autoencoder features
class AutoencoderFeatures(nn.Module):
    def __init__(self, in_channels=1536, latent_dim=50):
        super().__init__()

        # Definir números de canales intermedios
        channels_1 = (in_channels + 2 * latent_dim) // 2 # 818
        channels_2 = 2 * latent_dim # 100 (latent_dim: dimensión en la capa central)

        # Encoder: comprime el tensor de features de 1536 canales a 50
        self.encoder = nn.Sequential(
            # Capa 1
            nn.Conv2d(in_channels, channels_1, kernel_size=1),
            nn.BatchNorm2d(channels_1),
            nn.ReLU(inplace=True), # inplace=True para reducir uso de memoria

            # Capa 2
            nn.Conv2d(channels_1, channels_2, kernel_size=1),
            nn.BatchNorm2d(channels_2),
            nn.ReLU(inplace=True),

            # Bottleneck
            nn.Conv2d(channels_2, latent_dim, kernel_size=1)
        )

        # Decoder: reconstruye el feature de entrada. Es simplemente la arquitectura
        # del codificador pero en reversa

        self.decoder = nn.Sequential(
            # Capa 1
            nn.Conv2d(latent_dim, channels_2, kernel_size=1),
            nn.BatchNorm2d(channels_2),
            nn.ReLU(inplace=True),

            # Capa 2
            nn.Conv2d(channels_2, channels_1, kernel_size=1),
            nn.BatchNorm2d(channels_1),
            nn.ReLU(inplace=True),

            # Capa 3: salida reconstruida (no se usa función de activación
            # pues los features pueden incluso tener valores negativos)
            nn.Conv2d(channels_1, in_channels, kernel_size=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
x = torch.randn(1, 1536, 28, 28).to(device)
print(f'Tamaño entrada: {x.shape}')
modelo = AutoencoderFeatures().to(device)
features = modelo(x)
print(f'Tamaño salida: {features.shape}')

In [ ]:
summary(modelo, input_size=(1536, 28, 28))

In [ ]:
# Instancias del extractor y del autoencoder en la GPU
extractor = Extractor().to(device)
autoencoder = AutoencoderFeatures().to(device)

# Pérdida y optimizador
perdida = torch.nn.MSELoss()
optimizador = torch.optim.Adam(autoencoder.parameters(), lr=LR)

# Checkpoint para guardar el mejor modelo
checkpoint_dir = '/workspace/checkpoints_VAE'
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Los modelos se guardarán en: {checkpoint_dir}")

# Variable para rastrear la mejor pérdida
mejor_perdida = float('inf')

In [ ]:
print(f"Device extractor: {next(extractor.parameters()).device}")
print(f"Device autoencoder: {next(autoencoder.parameters()).device}")
print(f"VRAM ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print(f"Device: {device}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import time

start = time.time()
for i, (lote, _) in enumerate(dataloader_train_real):
    lote = lote.to(device)
    with torch.no_grad():
        features = extractor(lote)
    salida = autoencoder(features)
    elapsed = time.time() - start
    print(f"Batch {i+1}: {elapsed:.2f}s")
    start = time.time()
    if i >= 4:
        break

In [ ]:
# Entrenamiento
perdidas_por_iteracion = []
for epoch in tqdm(range(NITS)):
    autoencoder.train()
    perdida_total_iteracion = 0.0
    for lote, _ in dataloader_train_real:
        lote = lote.to(device)

        # Extraer features ***SIN ENTRENAR EL EXTRACTOR**
        with torch.no_grad():
            features = extractor(lote)

        # Pasar features por el autoencoder
        salida = autoencoder(features)
        
        # Calcular pérdida
        loss = perdida(salida, features)

        # Actualizar parámetros del autoencoder
        optimizador.zero_grad()
        loss.backward()
        optimizador.step()

        # Acumular pérdidas
        perdida_total_iteracion += loss.item()

    # Terminada la iteración, calcular la pérdida promedio
    perdida_promedio = perdida_total_iteracion / len(dataloader_train_real)
    perdidas_por_iteracion.append(perdida_promedio)

    # Imprimir info de esta iteración
    print(f"Fin de iteración {epoch + 1}/{NITS} - Pérdida Promedio: {perdida_promedio:.6f}")

    # Y guardar el mejor modelo hasta ahora (basado en la pérdida de la época)
    if perdida_promedio < mejor_perdida:
        mejor_perdida = perdida_promedio

        torch.save({
            'epoch': epoch,
            'model_state_dict': autoencoder.state_dict(),
            'optimizer_state_dict': optimizador.state_dict(),
            'loss': mejor_perdida,
        }, os.path.join(checkpoint_dir, 'mejor_modelo_cnn_vae.pth'))


In [ ]:
class CNNToVAEAdapter(nn.Module):
    """
    Convierte [B, 1536, 28, 28] → [B, 3, 512, 512]
    para que el encoder del VAE lo procese como si fuera una imagen.
    """
    def __init__(self, in_channels=1536, out_channels=3, target_size=256):
        super().__init__()

        self.conv_reduce = nn.Sequential(
            # Reducir canales progresivamente
            nn.Conv2d(1536, 512, kernel_size=3, padding=1),
            nn.GroupNorm(32, 512),
            nn.SiLU(),

            nn.Conv2d(512, 128, kernel_size=3, padding=1),
            nn.GroupNorm(32, 128),
            nn.SiLU(),

            nn.Conv2d(128, 3, kernel_size=1),  # → [B, 3, 28, 28]
        )

        # Upsample 28x28 → 512x512
        self.upsample = nn.Upsample(
            size=(target_size, target_size),
            mode='bilinear',
            align_corners=False
        )

        # Normalizar a [-1, 1] (rango esperado por el VAE)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.conv_reduce(x)   # [B, 3, 28, 28]
        x = self.upsample(x)      # [B, 3, 512, 512]
        x = self.tanh(x)          # [-1, 1]
        return x

In [ ]:
# Instancias del extractor y del autoencoder en la GPU
import os
extractor = Extractor().to(device)
autoencoder = AutoencoderFeatures().to(device)

# Cargar el mejor modelo
checkpoint = torch.load(os.path.join(checkpoint_dir, 'mejor_modelo_cnn_vae.pth'))
autoencoder.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# Llevar el modelo al modo "inferencia"

cnn_extractor = Extractor().to(device)
adapter = CNNToVAEAdapter().to(device)

cnn_extractor.eval()
adapter.eval()
autoencoder.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

y_true = []
errores = []
with torch.no_grad():
    for img, label in tqdm(dataloader_test):
        img = img.to(device)

        feature_cnn = cnn_extractor(img)
        adapted_feature = adapter(feature_cnn)
        recon = autoencoder(feature_cnn)

        flat_orig  = feature_cnn.flatten(start_dim=1)
        flat_recon = recon.flatten(start_dim=1)

        mse    = F.mse_loss(flat_orig, flat_recon, reduction='none').mean(dim=1)
        l1     = F.l1_loss(flat_orig, flat_recon, reduction='none').mean(dim=1)
        cosine = 1 - F.cosine_similarity(flat_orig, flat_recon, dim=1)  # disimilitud

        error_combinado = mse + l1 + cosine


        loss_per_image = error_combinado.view(error_combinado.shape[0], -1).mean(dim=1)

        errores.extend(loss_per_image.cpu().tolist())
        y_true.extend(label.tolist())

In [ ]:
errores = np.array(errores)
y_true = np.array(y_true)

plt.figure(figsize=(15, 5))

plt.hist(errores[y_true == 0], bins=300, alpha=0.5, label="fake")
plt.hist(errores[y_true == 1], bins=300, alpha=0.5, label="real")
plt.legend()
plt.title("Distribución de los errores de CNN + VAE")
plt.xlabel("Error de reconstrucción por similitud de coseno")
plt.ylabel("Conteo");
plt.savefig('/workspace/imagenes_metricas_vae/distribucion_errores_cnn_vae_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fake = 0
real = 1
etiquetas = test_ds.classes

# Rango de umbrales a considerar
umbral_min = np.min(errores[y_true==fake])
umbral_max = np.mean(errores[y_true==fake]) + 5*np.std(errores[y_true==fake])
print(umbral_min, umbral_max)
umbrales = np.linspace(umbral_min, umbral_max, 100)

# Iterar por cada umbral, calcular puntaje F1 y almacenar
scores = []
for umbral in umbrales:
    y_pred = [real if error < umbral else fake for error in errores]
    y_pred = np.array(y_pred)

    report = classification_report(y_true, y_pred, output_dict=True)

    macro_f1 = report['macro avg']['f1-score']
    scores.append(macro_f1)

# Escoger el mejor umbral, el que arroje un F1 máximo
mejor_idx = np.argmax(scores)
mejor_umbral = umbrales[mejor_idx]
mejor_f1 = scores[mejor_idx]

print(f"Mejor umbral: {mejor_umbral:.4f}")
print(f"Mejor puntaje F1: {mejor_f1:.4f}")

In [ ]:
print(f"Tamaño de errores: {len(errores)}")
print(f"Tamaño de categorias: {len(y_true)}")
print(f"Tamaño de y_pred: {len(y_pred)}")

In [ ]:
y_pred = [real if error < mejor_umbral else fake for error in errores]
y_pred = np.array(y_pred)

print("Distribución y_true:", np.bincount(y_true))
print("Distribución y_pred:", np.bincount(y_pred))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=etiquetas,
    yticklabels=etiquetas
)
plt.xlabel('Predicho', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title('Matriz de Confusión — CNN + VAE', fontsize=13)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/confusion_matrix_cnn_vae_final.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=etiquetas))

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=etiquetas, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()

fig, ax = plt.subplots(figsize=(8, 4)) 
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=df_report.values,
                 colLabels=df_report.columns,
                 rowLabels=df_report.index,
                 cellLoc='center',
                 loc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2) # Ajustar escala

plt.savefig('/workspace/imagenes_metricas_vae/classification_report_cnn_vae_final.png', bbox_inches='tight', dpi=300)
print("Imagen guardada como classification_report.png")

In [ ]:
fpr, tpr, _ = roc_curve(y_true, [-e for e in errores])
auc_score   = roc_auc_score(y_true, [-e for e in errores])

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'VAE (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC — CNN + VAE', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/workspace/imagenes_metricas_vae/roc_curve_cnn_vae_final.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC final (test completo): {auc_score:.4f}")